# ow-0009 -- export raw support/density grid data

Exports the actual gridded Support/KDE density values (max over the 72h backward
window) that the heatmap figures (e.g. `AIS_top15_tracks.png`,
`possible_source_origins_ais_attribution.png`) are rendered FROM -- as a flat
(lat, lon, density) table, so the underlying numbers are available outside the PNGs.
Same unmodified T14-T17 candidate sweep as every other notebook this session;
no new computation, only a new export of an already-computed quantity.

In [1]:
import os, sys, time, json as _json
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np
import pandas as pd

from iste.ensemble_run import EnsembleRunRecords
from iste.analysis_grid import generate_analysis_grid
from iste.candidate_extraction import CandidateExtractionConfig, find_candidate_peaks
from iste.kdtree_neighborhood import build_particle_neighborhood_index
from iste.particle_query import query_particle_slice
from iste.support_features import SupportFeatureConfig, compute_support_map

RESULT_DIR = os.path.join(REPO_ROOT, "result_ow0009")
OUTPUT_DIR = os.path.join(REPO_ROOT, "output", "ow-0009")
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [2]:
history_path = os.path.join(RESULT_DIR, "stage3_particle_history_ow0009.npz")
cached = np.load(history_path, allow_pickle=True)
def _none_if_empty(arr):
    return None if arr.size == 0 else arr
records = EnsembleRunRecords(
    particle_id=cached["particle_id"], forcing_member_id=cached["forcing_member_id"],
    observation_pixel_id=cached["observation_pixel_id"], timestamp=cached["timestamp"],
    lat=cached["lat"], lon=cached["lon"], log_weight=cached["log_weight"], valid=cached["valid"],
    death_time=_none_if_empty(cached["death_time"]), death_lat=_none_if_empty(cached["death_lat"]),
    death_lon=_none_if_empty(cached["death_lon"]), death_reason=_none_if_empty(cached["death_reason"]),
    survival_mark=_none_if_empty(cached["survival_mark"]),
)
T_OBS = np.datetime64(np.max(records.timestamp), "ns")
grid = generate_analysis_grid(T_OBS, window_hours=72.0, interval_hours=1.0)
historical_timestamps = grid[1:]

spill_data = _json.load(open(os.path.join(REPO_ROOT, "data", "database_payload.json"), encoding="utf-8"))
spill = next(s for s in spill_data["spills"] if s["image_filename"] == "ow-0009.jpg")
obs_bbox = spill["bounding_box"]
valid_mask = records.valid
lat_min = min(obs_bbox["upper_left"]["lat"], obs_bbox["bottom_right"]["lat"], float(records.lat[valid_mask].min()))
lat_max = max(obs_bbox["upper_left"]["lat"], obs_bbox["bottom_right"]["lat"], float(records.lat[valid_mask].max()))
lon_min = min(obs_bbox["upper_left"]["lon"], obs_bbox["bottom_right"]["lon"], float(records.lon[valid_mask].min()))
lon_max = max(obs_bbox["upper_left"]["lon"], obs_bbox["bottom_right"]["lon"], float(records.lon[valid_mask].max()))
PAD_DEG = 0.03
support_lat_grid = np.arange(lat_min - PAD_DEG, lat_max + PAD_DEG, 0.003)
support_lon_grid = np.arange(lon_min - PAD_DEG, lon_max + PAD_DEG, 0.003)
NEIGHBORHOOD_RADIUS_M = 1000.0
support_config = SupportFeatureConfig()
extraction_config = CandidateExtractionConfig(peak_neighborhood_cells=5, min_separation_m=1500.0, core_radius_m=1000.0)
lon_mesh, lat_mesh = np.meshgrid(support_lon_grid, support_lat_grid)

t0 = time.time()
support_grids_by_t_s = {}
for t_s in historical_timestamps:
    slice_ = query_particle_slice(records, t_s)
    index = build_particle_neighborhood_index(slice_)
    support_map = compute_support_map(index, lat_mesh.ravel(), lon_mesh.ravel(), radius_m=NEIGHBORHOOD_RADIUS_M, config=support_config)
    support_grids_by_t_s[np.datetime64(t_s, "ns")] = np.array([sf.support for sf in support_map]).reshape(len(support_lat_grid), len(support_lon_grid))

_t0_slice = query_particle_slice(records, T_OBS)
_t0_index = build_particle_neighborhood_index(_t0_slice)
_t0_support_map = compute_support_map(_t0_index, lat_mesh.ravel(), lon_mesh.ravel(), radius_m=NEIGHBORHOOD_RADIUS_M, config=support_config)
support_grids_by_t_s[T_OBS] = np.array([sf.support for sf in _t0_support_map]).reshape(len(support_lat_grid), len(support_lon_grid))
t1 = time.time()
print(f"T14-T17 sweep over {len(historical_timestamps)} historical timestamps + T0 took {t1-t0:.1f}s -- "
      f"grid {len(support_lat_grid)}x{len(support_lon_grid)}, {len(support_grids_by_t_s)} hourly grids")

T14-T17 sweep over 72 historical timestamps + T0 took 293.7s -- grid 215x288, 73 hourly grids


In [3]:
AGGREGATE_SUPPORT_GRID = np.maximum.reduce(list(support_grids_by_t_s.values()))
GLOBAL_SUPPORT_VMAX = float(AGGREGATE_SUPPORT_GRID.max())
DISPLAY_GAMMA = 0.4
display_normalized = (np.clip(AGGREGATE_SUPPORT_GRID, 0, None) / GLOBAL_SUPPORT_VMAX) ** DISPLAY_GAMMA

lat_flat, lon_flat = lat_mesh.ravel(), lon_mesh.ravel()
grid_df = pd.DataFrame({
    "lat": lat_flat, "lon": lon_flat,
    "support_density_raw": AGGREGATE_SUPPORT_GRID.ravel(),
    "support_density_display_normalized": display_normalized.ravel(),
})
grid_path = os.path.join(OUTPUT_DIR, "support_density_grid.csv")
grid_df.to_csv(grid_path, index=False)
print(f"saved {grid_path} ({len(grid_df)} rows -- {len(support_lat_grid)}x{len(support_lon_grid)} grid, "
      f"max-over-72h aggregate, same data used to render AIS_top15_tracks.png / "
      f"possible_source_origins_ais_attribution.png)")
print(grid_df.describe().to_string())

saved C:\Users\venka\OneDrive\Desktop\SIH-2026\project-v5\output\ow-0009\support_density_grid.csv (61920 rows -- 215x288 grid, max-over-72h aggregate, same data used to render AIS_top15_tracks.png / possible_source_origins_ais_attribution.png)
                lat           lon  support_density_raw  support_density_display_normalized
count  61920.000000  61920.000000         61920.000000                        61920.000000
mean      34.708955     35.208223            47.113707                            0.069398
std        0.186195      0.249416           174.326580                            0.150846
min       34.387955     34.777723             0.000000                            0.000000
25%       34.546955     34.992973             0.000000                            0.000000
50%       34.708955     35.208223             0.000000                            0.000000
75%       34.870955     35.423473             0.000000                            0.000000
max       35.029955     35.6

In [4]:
checks = []
checks.append(("grid row count matches lat x lon dims", len(grid_df) == len(support_lat_grid) * len(support_lon_grid)))
checks.append(("no negative density values", bool((grid_df['support_density_raw'] >= 0).all())))
checks.append(("display-normalized values in [0,1]", bool(grid_df['support_density_display_normalized'].between(0, 1).all())))
checks.append(("file saved", os.path.exists(grid_path)))
for label, ok in checks:
    print(f"[{'OK' if ok else 'FAIL'}] {label}")
assert all(ok for _, ok in checks)

[OK] grid row count matches lat x lon dims
[OK] no negative density values
[OK] display-normalized values in [0,1]
[OK] file saved
